Imports

In [1]:
import pandas as pd
import numpy as np
import os

Percentage threshold for dropping columns

In [2]:
MAX_MISSINGNESS_THRESHOLD = 20

Helper functions

In [ ]:
def _drop_duplicates_and_print_loss(df: pd.DataFrame):
    prev_num_rows = df.shape[0]
    df = df.drop_duplicates()
    new_num_rows = df.shape[0]

    print("Number of duplicate rows:", prev_num_rows - new_num_rows)
    return df


def _dropna_and_print_loss(df: pd.DataFrame, columns):
    prev_num_rows = df.shape[0]
    df = df.dropna(subset=columns)
    new_num_rows = df.shape[0]

    print(f"Number of rows with empty {columns}:", prev_num_rows - new_num_rows)
    return df


def _get_column_data_missingness(df: pd.DataFrame):
    missingness = df.isnull().mean() * 100
    return missingness.sort_values(ascending=False)


def _drop_highly_missing_columns(df: pd.DataFrame, columns = None):
    missingness = _get_column_data_missingness(df)
    columns_to_drop = missingness[missingness > MAX_MISSINGNESS_THRESHOLD].index 

    return df.drop(columns=columns_to_drop)


def _fill_missing_numeric_values(df: pd.DataFrame, columns=None, value=None):
    if not columns:
        columns = df.columns

    for col in columns:
        if df[col].dtype in [np.float64, np.int64]:

            if value is not None:
                df[col] = df[col].fillna(value)
                continue

            # Hierarchical mean imputation
            df[col] = df.groupby('Disaster Subtype')[col].transform(
                lambda x: x.fillna(x.mean())
            )

            df[col] = df.groupby('Disaster Type')[col].transform(
                lambda x: x.fillna(x.mean())
            )

            df[col] = df.groupby('Disaster Subgroup')[col].transform(
                lambda x: x.fillna(x.mean())
            )

            # fallback global
            df[col] = df[col].fillna(df[col].mean())

    return df


def _format_date(df: pd.DataFrame, prefix: str):
    return pd.to_datetime(df[prefix + ' Year'].astype(int).astype(str) + '-' + df[prefix + ' Month'].astype(int).astype(str) + '-' + df[prefix + ' Day'].astype(int).astype(str), format='%Y-%m-%d', errors='coerce')


def _number_of_days(df: pd.DataFrame):
    start_dates = _format_date(df, 'Start')
    end_dates = _format_date(df, 'End')
    return (end_dates - start_dates).dt.days


def _get_total_affected(df: pd.DataFrame):
    # Preserve the original dataset's definition of Total Affected:
    # No. Affected + No. Injured + No. Homeless

    return df['No. Affected'] + df['No. Injured'] + df['No. Homeless']

def _get_total_impacted(df: pd.DataFrame):
    return df['No. Affected'] + df['No. Injured'] + df['No. Homeless'] + df['Total Deaths']


Data preparation function

In [4]:
def prepare_emdat_data(df: pd.DataFrame):
    
    df = _drop_duplicates_and_print_loss(df)
    impact_cols = ['No. Injured', 'No. Affected', 'No. Homeless', 'Total Deaths']
    financial_damage_cols = ["Reconstruction Costs ('000 US$)", "Insured Damage ('000 US$)", "Total Damage ('000 US$)"]
    date_cols = ['Start Year', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day']

    mandatory_cols = ['DisNo.', 'ISO', 'Classification Key', 'Start Year']
    df = _dropna_and_print_loss(df, columns=mandatory_cols)
    df = _dropna_and_print_loss(df, date_cols)

    df['Number of Days'] = _number_of_days(df)
    df = _fill_missing_numeric_values(df, columns=['Number of Days'], value=0)
    df = _fill_missing_numeric_values(df, columns=impact_cols, value=0)
    df = _fill_missing_numeric_values(df, columns=financial_damage_cols, value=0)

    df['Total Affected'] = _get_total_affected(df).astype(int)
    df['Total Impacted'] = _get_total_impacted(df).astype(int)
    df['External IDs'] = df['External IDs'].str.split('|')
    df['Date'] = _format_date(df, 'Start')

    df = _drop_highly_missing_columns(df)
    df = _fill_missing_numeric_values(df)

    df['Fatality Rate'] = df['Total Deaths'] / df['Total Impacted']
    
    integer_cols = impact_cols + date_cols
    df[integer_cols] = df[integer_cols].apply(pd.to_numeric, errors='coerce').astype('Int64')

    df['Number of Days'] = df['Number of Days'].replace(0, 1)
    df = df.drop(columns=['Entry Date', 'Last Update', "Reconstruction Costs ('000 US$)"])

    return df

Execution

In [5]:
dir_path = r"..\data"
file_path = os.path.join(dir_path, "raw_public_emdat_2026.xlsx")
raw_data = pd.read_excel(file_path)
prepared_data = prepare_emdat_data(raw_data)
prepared_data.to_csv(os.path.join(dir_path, "prepared_public_emdat_2026.csv"), index=False)

Number of duplicate rows: 0
Number of rows with empty ['DisNo.', 'ISO', 'Classification Key', 'Start Year']: 0
Number of rows with empty ['Start Year', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day']: 1696
